In [0]:
/* ---------------------------------------------------------------------
   1) Funcao auxiliar: chave de comparacao de nomes (MAIUSCULAS, sem acento).
      Usada nos DOIS lados do JOIN medicamento x catalogo.
   --------------------------------------------------------------------- */
CREATE OR REPLACE FUNCTION anvisa.fn_chave_nome(texto STRING)
RETURNS STRING
COMMENT 'Chave de comparacao de nomes: maiusculas e sem acento.'
RETURN translate(upper(trim(texto)),
                 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ',
                 'AAAAAEEEEIIIIOOOOOUUUUC');

In [0]:
/* =====================================================================
   2) TABELA GOLD  anvisa.g_vigimed_obt_ml
   GRAO: 1 linha por notificacao (id_notificacao unico).

   ---------------------------------------------------------------------
   A) COLUNAS DESCARTADAS POR TEREM > 49% DE NULOS
      (% = nulos / linhas da Silver; chaves de JOIN e alvo ficam de fora da regra)

   s_vigimed_notificacoes (363.249 linhas) - 5 colunas
     idade_gestacional_momento_reacao 99,9% | notificacao_parent_child 99,7% |
     altura_cm 78,1% | peso_kg 60,9% | grupo_idade 57,3%

   s_vigimed_reacoes (1.027.408 linhas) - 5 colunas
     duracao_dias 83,3% | duracao_texto 83,3% | data_fim 70,5% |
     ind_data_fim_incompleta 70,5% | gravidade_criterios 68,2%

   s_vigimed_medicamentos (688.856 linhas) - 20 colunas
     via_administracao_mae_pai 99,98% | problemas_adicionais 96,2% |
     componente_suspeito 89,2% | duracao_dias 84,8% | duracao_texto 84,8% |
     concentracao 82,7% | detentor_registro 75,8% | data_fim_administracao 71,4% |
     ind_fim_incompleta 71,3% | via_administracao 71,3% | forma_farmaceutica 70,2% |
     numero_lote 68,9% | frequencia_dose 68,2% | dose_unidade 61,1% |
     dose_valor 60,3% | dose_texto 60,3% | indicacao_relatada 57,2% |
     indicacao_meddra 57,1% | data_inicio_administracao 52,8% | ind_inicio_incompleta 52,8%

   s_dados_abertos_medicamentos (42.741 linhas) - nenhuma

   Ficaram por pouco abaixo do corte (entram na Gold): data_notificacao 48,5% (nao
   usada, ver B), acao_adotada 48,7% e posologia 48,2%.
   (posologia e texto livre com ~168 mil valores distintos: avalie descartar no treino.)

   ---------------------------------------------------------------------
   B) OUTRAS COLUNAS FORA DA GOLD (nao e regra de nulos)
      - Auditoria/constantes: dt_ingestao_bronze, dt_processamento_silver,
        tipo_produto, ind_vencimento_so_mes.
      - Identificadores sem valor preditivo: numero_registro_produto,
        numero_processo, cnpj_empresa_detentora (o nome da empresa fica).
      - Redundantes: situacao_registro (= ind_registro_ativo) e principio_ativo do
        catalogo (ja existe principios_ativos do WHODrug).
      - Datas cruas (modelo nao consome DATE e imputar data seria inventar dado):
        so data_inclusao_sistema vira ano/mes. Fora: data_nascimento (a idade ja
        cobre), data_notificacao (~ data_inclusao_sistema), data_ultima_atualizacao
        (follow-up posterior ao evento), data_inicio + ind_data_inicio_incompleta
        (reacao), data_finalizacao_processo e data_vencimento_registro (catalogo).
      - Derivadas de datas descartadas (quase constantes em FALSE):
        ind_data_fim_anterior_inicio, ind_fim_anterior_inicio.

   ---------------------------------------------------------------------
   C) VARIAVEL ALVO ind_grave (INT 0/1, nunca nula)
      Prioridade:
        1) s_vigimed_notificacoes.ind_grave, se nao for nula;
        2) senao, s_vigimed_reacoes.ind_grave agregado por notificacao com MAX:
           grave se QUALQUER reacao for grave (mesma regra que a Silver usa para
           montar o ind_grave da notificacao, entao o rotulo imputado segue a
           mesma definicao do rotulo original);
        3) senao (nenhuma das duas fontes tem informacao) -> a notificacao e
           DESCARTADA da Gold: ausencia de rotulo nao e "nao grave".
      Medido na carga de 15/11/2026 (Silver com 363.249 notificacoes):
        - 85.018 notificacoes sem ind_grave; a regra 2 rotulou 2.184 (472 nao
          graves e 1.712 graves); as 82.834 restantes (22,8%) saem da Gold,
          que fica com 280.415 linhas.
        - Entre as rotuladas, 54,2% sao graves (54,1% so nas de origem
          'notificacao'): a classe e praticamente balanceada, NAO minoritaria.
        - Por que descartar em vez de imputar 0: o 0 sem evidencia criaria
          falsos "nao graves" (estimativa: ~45 mil, se o campo faltasse ao
          acaso) e o modelo aprenderia o padrao de preenchimento em vez da
          gravidade clinica.
      BALANCEAMENTO: com ~54/46 nao ha necessidade de class_weight nem de
      sobreamostragem; se usar, so no conjunto de TREINO (a Gold nao balanceia).
      origem_ind_grave ('notificacao' | 'reacoes') so serve para auditoria:
      NAO usar como feature (vazamento do alvo). Avalie o modelo principalmente
      nas linhas com origem 'notificacao'.

   ---------------------------------------------------------------------
   D) CHAVES DE JOIN (notificacoes e a base: tem o perfil do paciente e o alvo)
      - notificacoes -> reacoes       : id_notificacao   (1:N -> reduzido a 1:1)
      - notificacoes -> medicamentos  : id_notificacao   (1:N -> reduzido a 1:1)
      - medicamentos -> catalogo      : nome normalizado (N:1 apos deduplicar). O VigiMed
        nao traz numero de registro, entao a unica ponte e
        nome_medicamento (WHODrug) = nome_produto (catalogo), via fn_chave_nome.
      Todos sao LEFT JOIN: nenhuma notificacao se perde por falta de reacao,
      medicamento ou catalogo (a unica exclusao e a falta de rotulo, ver C).
      Ids que existem so em reacoes/medicamentos
      (22.303 / 2.940) ficam de fora: sem notificacao nao ha perfil nem alvo primario.
   ===================================================================== */

CREATE OR REPLACE TABLE anvisa.g_vigimed_obt_ml
USING DELTA
-- LOCATION 'abfss://projetos@stmarcelodatalab.dfs.core.windows.net/hands_on_mackenzie/03_gold/g_vigimed_obt_ml'
COMMENT 'Gold VigiMed (OBT): 1 linha por notificacao com rotulo, sem nulos, alvo binario ind_grave (0/1).'
AS
WITH

/* 1. reacao: 1 linha por notificacao.
   Reacao principal = QUALIFY ROW_NUMBER = 1. Desempate: inicio mais antigo
   (nulo por ultimo), depois reacao_pt, reacao_llt, desfecho.
   De proposito NAO se ordena por gravidade (a reacao "grave" como principal
   vazaria o alvo para as features).
   As janelas COUNT/MAX rodam ANTES do QUALIFY, ou seja, sobre TODAS as reacoes. */
reacao AS (
  SELECT
    id_notificacao,
    reacao_llt, reacao_pt, reacao_hlt, reacao_hlgt, reacao_soc, desfecho,
    COUNT(*)                        OVER (PARTITION BY id_notificacao) AS qtd_reacoes,
    MAX(CAST(ind_grave AS INT))     OVER (PARTITION BY id_notificacao) AS ind_grave_reacoes  -- NULL se todas nulas
  FROM anvisa.s_vigimed_reacoes
  QUALIFY ROW_NUMBER() OVER (
            PARTITION BY id_notificacao
            ORDER BY data_inicio ASC NULLS LAST, reacao_pt, reacao_llt, desfecho
          ) = 1
),

/* 2. medicamento: 1 linha por notificacao.
   Medicamento principal = QUALIFY ROW_NUMBER = 1. Desempate: 'Suspeito' primeiro,
   depois 'Interagente', depois os demais (ex.: 'Concomitante'); depois nome,
   acao_adotada e posologia. qtd_medicamentos conta todas as linhas (polimedicacao).
   Listas viram valor simples (a Gold nao tem ARRAY): principios_ativos -> 'A | B';
   codigos_atc -> 1o codigo (try_element_at devolve NULL se a lista for vazia). */
medicamento AS (
  SELECT
    id_notificacao,
    relacao_medicamento_evento,
    nome_medicamento,
    NULLIF(array_join(principios_ativos, ' | '), '') AS principios_ativos,
    try_element_at(codigos_atc, 1)                   AS codigo_atc,
    acao_adotada,
    posologia,
    anvisa.fn_chave_nome(nome_medicamento)           AS chave_nome,
    COUNT(*) OVER (PARTITION BY id_notificacao)      AS qtd_medicamentos
  FROM anvisa.s_vigimed_medicamentos
  QUALIFY ROW_NUMBER() OVER (
            PARTITION BY id_notificacao
            ORDER BY CASE relacao_medicamento_evento
                       WHEN 'Suspeito'    THEN 1
                       WHEN 'Interagente' THEN 2
                       ELSE 3
                     END,
                     nome_medicamento, acao_adotada, posologia
          ) = 1
),

/* 3. catalogo: 1 linha por nome de produto.
   O catalogo repete o mesmo nome (varios fabricantes/apresentacoes); sem isto o
   JOIN multiplicaria as notificacoes. Desempate: registro ATIVO primeiro,
   vencimento mais distante, finalizacao mais recente, numero do registro. */
catalogo AS (
  SELECT
    anvisa.fn_chave_nome(nome_produto) AS chave_nome,
    categoria_regulatoria,
    classe_terapeutica,
    empresa_detentora_registro,
    ind_registro_ativo
  FROM anvisa.s_dados_abertos_medicamentos
  QUALIFY ROW_NUMBER() OVER (
            PARTITION BY anvisa.fn_chave_nome(nome_produto)
            ORDER BY ind_registro_ativo DESC,
                     data_vencimento_registro  DESC NULLS LAST,
                     data_finalizacao_processo DESC NULLS LAST,
                     numero_registro_produto
          ) = 1
),

/* 4. unificada: JOINs (chaves descritas em D). Cada lado ja tem 1 linha por chave,
   entao o resultado continua com 1 linha por notificacao. Os dois ind_grave de
   origem entram com nomes distintos para a CTE do alvo escolher. */
unificada AS (
  SELECT
    n.id_notificacao,
    n.ind_grave AS ind_grave_notificacao,
    r.ind_grave_reacoes,
    n.uf, n.tipo_entrada_vigimed, n.recebido_de, n.tipo_notificacao, n.sexo, n.notificador,
    n.ind_gestante, n.ind_lactante, n.idade_momento_reacao, n.data_inclusao_sistema,
    r.qtd_reacoes, r.reacao_llt, r.reacao_pt, r.reacao_hlt, r.reacao_hlgt, r.reacao_soc, r.desfecho,
    m.qtd_medicamentos, m.relacao_medicamento_evento, m.nome_medicamento, m.principios_ativos,
    m.codigo_atc, m.acao_adotada, m.posologia,
    CASE WHEN c.chave_nome IS NULL THEN 0 ELSE 1 END AS ind_medicamento_no_catalogo,
    c.categoria_regulatoria, c.classe_terapeutica, c.empresa_detentora_registro, c.ind_registro_ativo
  FROM anvisa.s_vigimed_notificacoes n
  LEFT JOIN reacao      r ON r.id_notificacao = n.id_notificacao
  LEFT JOIN medicamento m ON m.id_notificacao = n.id_notificacao
  LEFT JOIN catalogo    c ON c.chave_nome     = m.chave_nome
),

/* 5. alvo: ind_grave binario (regra C). COALESCE respeita a ordem de prioridade:
   notificacao -> reacoes. O WHERE descarta a notificacao sem rotulo nas duas
   fontes (ausencia de informacao nao e "nao grave"), entao ind_grave nunca e nulo. */
alvo AS (
  SELECT
    *,
    COALESCE(CAST(ind_grave_notificacao AS INT), ind_grave_reacoes) AS ind_grave,
    CASE WHEN ind_grave_notificacao IS NOT NULL THEN 'notificacao' ELSE 'reacoes' END AS origem_ind_grave
  FROM unificada
  WHERE ind_grave_notificacao IS NOT NULL OR ind_grave_reacoes IS NOT NULL
),

/* 6. deduplicada: trava final de granularidade. As CTEs 1 a 4 ja entregam 1 linha
   por id_notificacao; se algum JOIN futuro multiplicar linhas, sobra 1 por
   notificacao (a mais completa: mais reacoes e mais medicamentos; depois ordem
   alfabetica para ser deterministico). */
deduplicada AS (
  SELECT *
  FROM alvo
  QUALIFY ROW_NUMBER() OVER (
            PARTITION BY id_notificacao
            ORDER BY qtd_reacoes DESC, qtd_medicamentos DESC, reacao_pt, nome_medicamento
          ) = 1
)

/* 7. SELECT final: imputacao (ZERO NULOS)
   - texto            -> 'Nao Informado'
   - flags (BOOLEAN)  -> INT 0/1; desconhecido = 0
   - contagens        -> 0 (notificacao sem reacao/medicamento tem 0 mesmo)
   - idade            -> mediana (0 anos seria uma idade valida e enganaria o modelo)
   - ano/mes          -> 0 (data_inclusao_sistema nao tem nulos; o COALESCE e so seguranca)
   - catalogo         -> 'Nao Informado' / 0 quando nao casou; ind_medicamento_no_catalogo
                         diferencia "registro inativo" de "sem correspondencia" */
SELECT
  id_notificacao,
  ind_grave,
  origem_ind_grave,

  -- notificacao
  COALESCE(uf,                   'Nao Informado') AS uf,
  COALESCE(tipo_entrada_vigimed, 'Nao Informado') AS tipo_entrada_vigimed,
  COALESCE(recebido_de,          'Nao Informado') AS recebido_de,
  COALESCE(tipo_notificacao,     'Nao Informado') AS tipo_notificacao,
  COALESCE(sexo,                 'Nao Informado') AS sexo,
  COALESCE(notificador,          'Nao Informado') AS notificador,
  COALESCE(CAST(ind_gestante AS INT), 0)          AS ind_gestante,
  COALESCE(CAST(ind_lactante AS INT), 0)          AS ind_lactante,
  CAST(COALESCE(idade_momento_reacao,
                (SELECT percentile_approx(idade_momento_reacao, 0.5)
                 FROM anvisa.s_vigimed_notificacoes)) AS DOUBLE) AS idade_momento_reacao,
  COALESCE(year(data_inclusao_sistema),  0)       AS ano_inclusao_sistema,
  COALESCE(month(data_inclusao_sistema), 0)       AS mes_inclusao_sistema,

  -- reacao principal
  CAST(COALESCE(qtd_reacoes, 0) AS INT)           AS qtd_reacoes,
  COALESCE(reacao_llt,  'Nao Informado')          AS reacao_llt,
  COALESCE(reacao_pt,   'Nao Informado')          AS reacao_pt,
  COALESCE(reacao_hlt,  'Nao Informado')          AS reacao_hlt,
  COALESCE(reacao_hlgt, 'Nao Informado')          AS reacao_hlgt,
  COALESCE(reacao_soc,  'Nao Informado')          AS reacao_soc,
  COALESCE(desfecho,    'Nao Informado')          AS desfecho,

  -- medicamento principal
  CAST(COALESCE(qtd_medicamentos, 0) AS INT)      AS qtd_medicamentos,
  COALESCE(relacao_medicamento_evento, 'Nao Informado') AS relacao_medicamento_evento,
  COALESCE(nome_medicamento,           'Nao Informado') AS nome_medicamento,
  COALESCE(principios_ativos,          'Nao Informado') AS principios_ativos,
  COALESCE(codigo_atc,                 'Nao Informado') AS codigo_atc,
  COALESCE(acao_adotada,               'Nao Informado') AS acao_adotada,
  COALESCE(posologia,                  'Nao Informado') AS posologia,

  -- catalogo de registros
  ind_medicamento_no_catalogo,
  COALESCE(categoria_regulatoria,      'Nao Informado') AS categoria_regulatoria,
  COALESCE(classe_terapeutica,         'Nao Informado') AS classe_terapeutica,
  COALESCE(empresa_detentora_registro, 'Nao Informado') AS empresa_detentora_registro,
  COALESCE(CAST(ind_registro_ativo AS INT), 0)          AS ind_registro_ativo
FROM deduplicada;

num_affected_rows,num_inserted_rows


## 3. Validacoes

In [0]:
-- A) GRANULARIDADE E ALVO: linhas = ids_distintos; alvo_invalido = 0;
--    descartadas_sem_rotulo = notificacoes da Silver que ficaram de fora (82.834)
SELECT count(*)                                                      AS linhas,
       count(DISTINCT id_notificacao)                                AS ids_distintos,
       (SELECT count(*) FROM anvisa.s_vigimed_notificacoes) - count(*) AS descartadas_sem_rotulo,
       count_if(ind_grave IS NULL OR ind_grave NOT IN (0, 1))        AS alvo_invalido
FROM anvisa.g_vigimed_obt_ml;

linhas,ids_distintos,descartadas_sem_rotulo,alvo_invalido
280415,280415,82834,0


In [0]:
%python

# B --> VALORES NULOS

df = spark.sql("select * from anvisa.g_vigimed_obt_ml")
       
for column in df.columns:
    print(column, df[df[column].isNull()].count())

id_notificacao 0
ind_grave 0
origem_ind_grave 0
uf 0
tipo_entrada_vigimed 0
recebido_de 0
tipo_notificacao 0
sexo 0
notificador 0
ind_gestante 0
ind_lactante 0
idade_momento_reacao 0
ano_inclusao_sistema 0
mes_inclusao_sistema 0
qtd_reacoes 0
reacao_llt 0
reacao_pt 0
reacao_hlt 0
reacao_hlgt 0
reacao_soc 0
desfecho 0
qtd_medicamentos 0
relacao_medicamento_evento 0
nome_medicamento 0
principios_ativos 0
codigo_atc 0
acao_adotada 0
posologia 0
ind_medicamento_no_catalogo 0
categoria_regulatoria 0
classe_terapeutica 0
empresa_detentora_registro 0
ind_registro_ativo 0


In [0]:
-- C) BALANCEAMENTO DO ALVO: % de graves no total e por origem do rotulo
SELECT origem_ind_grave, ind_grave, count(*) AS qtd,
       round(100 * count(*) / sum(count(*)) OVER (), 2) AS pct_total
FROM anvisa.g_vigimed_obt_ml
GROUP BY origem_ind_grave, ind_grave
ORDER BY origem_ind_grave, ind_grave;

origem_ind_grave,ind_grave,qtd,pct_total
notificacao,0,127823,45.58
notificacao,1,150408,53.64
reacoes,0,472,0.17
reacoes,1,1712,0.61


In [0]:
%python

import pandas

df = spark.sql("select * from anvisa.g_vigimed_obt_ml")

df.toPandas().to_csv("anvisa.g_vigimed_obt_ml.csv", header=True, index=False, encoding="utf-8")

In [0]:
%python
import pandas as pd
import csv

# 1. Carrega o dataframe do Spark (garantidamente sem nulos)
df = spark.sql("select * from anvisa.g_vigimed_obt_ml")

# 2. Converte para Pandas
pdf = df.toPandas()

# 3. Tratamento apenas para o erro "embedded nul(s)" no R:
# Remove o caractere invisível (\x00) das strings que interrompe a leitura
pdf = pdf.replace('\x00', '', regex=True)

# 4. Salva o CSV com segurança estrutural
pdf.to_csv(
    "anvisa.g_vigimed_obt_ml.csv", 
    header=True, 
    index=False, 
    encoding="utf-8",
    sep=",",
    quoting=csv.QUOTE_ALL,    # Coloca tudo entre aspas. Garante que vírgulas no texto não virem novas colunas.
    escapechar="\\"           # Protege aspas que já existam dentro do texto original.
)

In [0]:
%python
import json
from pyspark.sql import functions as F

def gerar_perfil_tabela(nome_tabela):
    print(f"Iniciando o perfil da tabela: {nome_tabela}")
    df = spark.table(nome_tabela)

    # 1. Agregação eficiente dos valores distintos e nulos (dispara apenas 1 Job)
    exprs = []
    for c in df.columns:
        exprs.append(F.approx_count_distinct(F.col(c)).alias(f"{c}_distinct"))
        exprs.append(F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_nulls"))

    agg_results = df.agg(*exprs).collect()[0].asDict()

    # 2. Montar dicionário estruturado para a LLM
    perfil_tabela = {
        "tabela": nome_tabela,
        "dicionario_de_dados": [
            {
                "coluna": col,
                "tipo_de_dado": dtype,
                "valores_distintos_aprox": agg_results[f"{col}_distinct"],
                "quantidade_nulos": agg_results[f"{col}_nulls"]
            }
            for col, dtype in df.dtypes
        ],
        "amostra_dados": [row.asDict() for row in df.limit(5).collect()]
    }

    # 3. Formatar em JSON
    # O uso de default=str é vital para evitar erros com datas ou decimais do PySpark
    resultado_llm = json.dumps(perfil_tabela, indent=2, default=str, ensure_ascii=False)

    # 4. Salvar o resultado em um arquivo JSON dinâmico
    # Substituímos pontos por underlines para evitar problemas com o nome do arquivo
    nome_arquivo_seguro = nome_tabela.replace(".", "_")
    nome_arquivo = f"{nome_arquivo_seguro}_schema.json"
    
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        f.write(resultado_llm)

    print(f"Arquivo '{nome_arquivo}' salvo com sucesso!\n")

# Lista de tabelas que você deseja processar
tabelas = [
    "anvisa.g_vigimed_obt_ml"
]

# Executa a função para cada tabela na lista
for tb in tabelas:
    gerar_perfil_tabela(tb)

Iniciando o perfil da tabela: anvisa.g_vigimed_obt_ml
Arquivo 'anvisa_g_vigimed_obt_ml_schema.json' salvo com sucesso!

